In [ ]:
input_file = "/Users/haroyang/Desktop/Project/tf/甄选官处理+聚类/0220甄选聚类/148b671d6275498a87966d5f763d6028.txt"

# 第一步：处理Gbot对话记录
选取用户对话，按openid拼接

In [ ]:
import pandas as pd
from datetime import datetime

def process_messages(input_file, output_file):
    """
    处理Excel/TXT文件中的消息数据
    1. 只保留用户消息
    2. 过滤掉自动回复
    3. 按用户和时间顺序合并消息（使用\n连接）
    4. 输出到新文件
    """
    # 根据文件扩展名选择不同的读取方式
    if input_file.endswith('.xlsx'):
        df = pd.read_excel(input_file)
    else:
        # 读取txt文件，跳过有问题的行
        df = pd.read_csv(input_file, sep='\t', on_bad_lines='skip')
    
    # 1. 只保留用户消息
    df = df[df['msg_type'] == '用户消息']
    
    # 2. 过滤掉自动回复
    df = df[~df['msg1'].str.contains(r'^\[自动回复\]', regex=True, na=False)]
    
    # 确保时间列格式正确
    df['operation_time'] = pd.to_datetime(df['operation_time'])
    
    # 3. 按用户ID和游戏ID分组，并按时间顺序合并消息
    def combine_messages(group):
        # 按时间排序
        sorted_group = group.sort_values('operation_time')
        # 合并消息，用换行符连接
        messages = '\n'.join(sorted_group['msg1'].astype(str))
        # 获取第一条消息的时间
        first_time = sorted_group['operation_time'].iloc[0]
        # 获取最后一条消息的时间
        last_time = sorted_group['operation_time'].iloc[-1]
        
        return pd.Series({
            'first_message_time': first_time,
            'last_message_time': last_time,
            'combined_messages': messages,
            'message_count': len(group)
        })
    
    # 按用户和游戏分组并合并消息
    result = df.groupby(['open_id', 'game_id']).apply(combine_messages).reset_index()
    
    # 4. 输出到新的Excel文件
    result.to_excel(output_file, index=False)
    
    return result


output_file_1 = input_file.replace('.txt', '_processed.xlsx')

try:
    result = process_messages(input_file, output_file_1)
    print(f"处理完成！共处理了 {len(result)} 个用户的消息")
    print(f"结果已保存到 {output_file_1}")
except Exception as e:
    print(f"处理过程中出现错误: {str(e)}")

# 第二步：Token计数
计算总Token量，并去除大于512Token的query

In [ ]:
import tiktoken  # 用于估算 token 数量

# 文件路径
input_file_path = output_file_1

# 读取输入文件并丢弃 combined_messages 为空的行
df = pd.read_excel(input_file_path)
df = df.dropna(subset=['combined_messages'])  # 丢弃 combined_messages 为空的行

# 初始化 tiktoken 编码器（根据使用的模型选择编码器）
def estimate_token_count(text, model="text-embedding-3-small"):
    encoding = tiktoken.encoding_for_model(model)
    return len(encoding.encode(text))

# 为每一条记录计算 token 数
df['token_count'] = df['combined_messages'].apply(lambda x: estimate_token_count(x))

# 筛选出 token 数不超过 512 的消息
df_filtered = df[df['token_count'] <= 512]

# 统计总 token 数和平均 token 数
total_tokens = df_filtered['token_count'].sum()
average_tokens = df_filtered['token_count'].mean()
max_tokens = df_filtered['token_count'].max()
min_tokens = df_filtered['token_count'].min()
percentile_95 = df_filtered['token_count'].quantile(0.99)

# 输出统计结果
print(f"筛选前消息总条数：{len(df)}")
print(f"筛选后消息总条数：{len(df_filtered)}")
print(f"被过滤掉的消息数量：{len(df) - len(df_filtered)}")
print(f"总 Token 数量：{total_tokens}")
print(f"平均每条消息 Token 数：{average_tokens:.2f}")
print(f"最大单条消息 Token 数：{max_tokens}")
print(f"最小单条消息 Token 数：{min_tokens}")
print(f"95%分位数 Token 数：{percentile_95:.2f}")

# 将筛选后的结果保存到新文件
output_file_2 = input_file_path.replace('.xlsx', '_token.xlsx')
df_filtered.to_excel(output_file_2, index=False)
print(f"\n筛选后的结果已保存至：{output_file_2}")

# 提取特征
使用阿里云embedding服务提取特征

In [ ]:
import json
import os
from openai import OpenAI
import pandas as pd
from tqdm import tqdm

# 创建 OpenAI 客户端
client = OpenAI(
    api_key=os.getenv("ALIYUN_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

# 文件路径
input_file_path = output_file_2
output_file_path_3= input_file_path.replace('.xlsx', '_embeddings.json')

# 读取输入文件并丢弃 combined_messages 为空的行
df = pd.read_excel(input_file_path)
df = df.dropna(subset=['combined_messages'])  # 丢弃 combined_messages 为空的行

# 检查是否有已存在的输出文件
if os.path.exists(output_file_path_3):
    processed_data = []
    processed_open_ids = set()
    with open(output_file_path_3, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                entry = json.loads(line.strip())
                processed_data.append(entry)
                processed_open_ids.add(entry['open_id'])
            except json.JSONDecodeError:
                continue
else:
    processed_data = []
    processed_open_ids = set()

# 筛选未处理的记录
df_to_process = df[~df['open_id'].isin(processed_open_ids)]
print(f"已处理 {len(processed_open_ids)} 条记录，剩余 {len(df_to_process)} 条记录待处理。")

# 定义生成嵌入的函数
# 修改生成嵌入的函数
def get_embedding(client, text, model="text-embedding-v3"):
    try:
        response = client.embeddings.create(
            input=text,
            model=model,
            dimensions=1024,
            encoding_format="float"
        )
        return response.data[0].embedding
    except Exception as e:
        print(f"获取嵌入向量时出错：{e}")
        return None

# 实时保存嵌入
successful_ids = set()  # 用于记录成功处理的ID

with open(output_file_path_3, 'a', encoding='utf-8') as f:
    for _, row in tqdm(df_to_process.iterrows(), total=len(df_to_process), desc="生成嵌入中"):
        try:
            embedding = get_embedding(client, row['combined_messages'])
            if embedding is not None:  # 只有成功获取embedding才保存
                entry = {
                    'open_id': row['open_id'],
                    'combined_messages': row['combined_messages'],
                    'embedding': embedding
                }
                f.write(json.dumps(entry, ensure_ascii=False) + '\n')
                successful_ids.add(row['open_id'])  # 记录成功处理的ID
        except Exception as e:
            print(f"处理 {row['open_id']} 时出错：{e}")
            continue

print(f"嵌入结果已实时保存至 {output_file_path_3}")
print(f"本次成功处理 {len(successful_ids)} 条记录")


# 聚类
使用k-means算法对未标注数据分组，每组称为一个簇。需预定义簇的数量。

In [ ]:
from sklearn.cluster import KMeans
import numpy as np
import json
import pandas as pd

# 文件路径
embeddings_file = output_file_path_3
output_file_4 = embeddings_file.replace('.json', '_clustered.xlsx')

# 加载 JSON 文件
def load_json(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                entry = json.loads(line.strip())
                data.append(entry)
            except json.JSONDecodeError as e:
                print(f"警告：跳过无效JSON行: {e}")
                continue
    return data

# 加载无标注数据
data = load_json(embeddings_file)

# 提取无标注数据的嵌入
embeddings = np.array([entry['embedding'] for entry in data])

# Step 1: KMeans 聚类
n_clusters = 25  # 聚类数设置为 20
print(f"开始 KMeans 聚类，聚类数设置为 {n_clusters}...")
kmeans = KMeans(n_clusters=n_clusters, n_init=30, random_state=42)  # 初始化 10 次以提高稳定性
kmeans.fit(embeddings)

# 获取聚类结果
cluster_labels = kmeans.labels_
print("KMeans 聚类完成！")

# Step 2: 保存聚类结果
results = []
for i, entry in enumerate(data):
    result = {
        'open_id': entry['open_id'],
        'combined_messages': entry['combined_messages'],
        'predicted_cluster': int(cluster_labels[i])  # KMeans 的聚类标签
    }
    results.append(result)

# 转换为 DataFrame
df_results = pd.DataFrame(results)

# 保存为 Excel 文件
df_results.to_excel(output_file_4, index=False)
print(f"聚类结果已保存至 {output_file_4}")


# 统计结果

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']  # Mac系统自带的字体
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

# 文件路径
input_excel_file = output_file_4
output_excel_file = input_excel_file.replace('.xlsx', '_category.xlsx')

# Step 1: 读取 Excel 文件
df = pd.read_excel(input_excel_file)

# 检查是否有需要的列
if 'predicted_cluster' not in df.columns:
    raise ValueError("Excel 文件中没有 'predicted_cluster' 列，请检查文件内容。")

# Step 2: 手动定义类别映射
category_mapping = {
    "无意义对话": [6, 19],  # 嗯、666
    "新年快乐": [3],  # 新年快乐
    "身份确认": [0, 2, 4, 7, 24, 20],  # "？"、建联场景、身份询问、你是？、有什么事、你好
    "真人？": [ 11, 15],  # 人机？、真人？身份？
    "骗子": [9],  # 骗子
    "王者口令": [1, 12],  # 两个都是王者口令
    "游戏吐槽/反馈": [21],  # 游戏信息询问
    "谢谢": [8, 17],  # 两个都是谢谢
    "负面情绪": [23],  # 骗子、滚
    "待细分-不回复策略": [10],  # 比较杂：辱骂、色情、情绪发泄、处对象
    "消费相关": [13, 16],  # 充值问题、消费
    "不需要": [22],  # 不需要
    "游戏资讯": [5, 18,14],  # 皮肤、火影资讯、返场
    
}

# 将类别映射转为数字到类别的反向映射
cluster_to_category = {}
for category, clusters in category_mapping.items():
    for cluster in clusters:
        cluster_to_category[cluster] = category

# 添加类别列
df['category'] = df['predicted_cluster'].map(cluster_to_category)

# Step 3: 按类别统计频次
category_counts = df['category'].value_counts().reset_index()
category_counts.columns = ['category', 'count']

# 计算比例
total_count = category_counts['count'].sum()
category_counts['proportion (%)'] = (category_counts['count'] / total_count * 100).round(2)  # 保留两位小数

# Step 4: 保存统计结果和样例到 Excel 文件
with pd.ExcelWriter(output_excel_file) as writer:
    # 保存原始数据
    df.to_excel(writer, sheet_name='Cluster Data', index=False)
    
    # 保存统计结果和样例到同一个sheet
    category_counts.to_excel(writer, sheet_name='Category Analysis', index=False)
    
    # 为每个类别抽样
    start_row = len(category_counts) + 3
    for category in category_counts['category']:
        # 获取该类别的所有样本
        category_df = df[df['category'] == category]
        
        # 先从每个cluster中抽取样本
        samples = []
        for cluster in category_mapping[category]:
            cluster_samples = category_df[category_df['predicted_cluster'] == cluster]
            if not cluster_samples.empty:
                sample = cluster_samples.sample(n=1, random_state=43)['combined_messages'].iloc[0]
                samples.append(f"Cluster {cluster}: {sample}")
        
        # 如果样本数不足5个，从整个类别中随机补充
        if len(samples) < 8:
            remaining_samples = category_df.sample(
                n=min(8-len(samples), len(category_df)), 
                random_state=43
            )['combined_messages'].tolist()
            samples.extend([f"补充样本: {sample}" for sample in remaining_samples])
        
        # 写入类别标题和样例
        pd.DataFrame([f"{category} 样例："]).to_excel(
            writer, 
            sheet_name='Category Analysis',
            startrow=start_row,
            header=False,
            index=False
        )
        
        pd.DataFrame(samples).to_excel(
            writer, 
            sheet_name='Category Analysis',
            startrow=start_row + 1,
            header=False,
            index=False
        )
        
        start_row += len(samples) + 3

print(f"统计结果已保存至 {output_excel_file}")

# Step 5: 绘制饼状图
# 定义颜色列表（可扩展或使用自动调色板）
colors = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99', '#c2c2f0', '#ffb3e6', '#ff6666', '#c2f0c2', '#c2f0f0', '#ffccff', '#ffcc99', '#c2f0c2']

plt.figure(figsize=(8, 8))  # 设置画布大小
plt.pie(
    category_counts['count'],
    labels=category_counts['category'],
    autopct='%1.1f%%',  # 显示百分比
    startangle=90,  # 起始角度
    colors=colors[:len(category_counts)],  # 为每个类别设置颜色
    textprops={'fontsize': 12}  # 文本大小
)
plt.title('类别分布统计', fontsize=16)  # 饼状图标题
plt.axis('equal')  # 保证饼图是圆形
plt.show()
